# A/B Testing & Experimentation Analytics
**Goal:** determine whether the redesigned checkout increases conversion.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import proportions_ztest, chi2_contingency
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/ab_test_users.csv')
df.head()

## 1. Data Quality & Experiment Balance

In [ ]:
print(df.shape)
print(df.isna().sum())
print(df['experiment_group'].value_counts())
print(pd.crosstab(df['experiment_group'], df['device'], normalize='index').round(3))

## 2. Primary Metric: Conversion Rate

In [ ]:
summary = df.groupby('experiment_group', observed=True)['converted'].agg(['count','sum','mean'])
summary['conversion_rate_pct'] = summary['mean']*100
summary

## 3. Hypothesis Test
H0: treatment <= control.  
H1: treatment > control.

In [ ]:
control = df.loc[df.experiment_group=='control','converted']
treatment = df.loc[df.experiment_group=='treatment','converted']
z, p = proportions_ztest([treatment.sum(), control.sum()], [len(treatment), len(control)], alternative='larger')
control_rate, treatment_rate = control.mean(), treatment.mean()
absolute_lift = treatment_rate-control_rate
relative_lift = absolute_lift/control_rate
print(f'z = {z:.3f}, p-value = {p:.6g}')
print(f'Absolute lift = {absolute_lift:.4%}')
print(f'Relative lift = {relative_lift:.2%}')

## 4. Chi-Square Cross-Check

In [ ]:
table = pd.crosstab(df.experiment_group, df.converted)
chi2, chi_p, dof, expected = chi2_contingency(table)
print(table)
print(f'chi2={chi2:.3f}, p-value={chi_p:.6g}, dof={dof}')

## 5. Bootstrap Confidence Interval

In [ ]:
rng = np.random.default_rng(42)
B = 5000
boot = np.empty(B)
for i in range(B):
    c = rng.choice(control, len(control), replace=True).mean()
    t = rng.choice(treatment, len(treatment), replace=True).mean()
    boot[i] = t-c
np.quantile(boot, [0.025, 0.975])

## 6. Segment Analysis

In [ ]:
segment = df.groupby(['device','experiment_group'], observed=True)['converted'].mean().unstack()
segment['absolute_lift'] = segment['treatment'] - segment['control']
segment.sort_values('absolute_lift', ascending=False)

## 7. Revenue Impact

In [ ]:
revenue = df.groupby('experiment_group', observed=True)['revenue'].agg(['mean','sum'])
revenue

## 8. Decision
Use the p-value and confidence interval for statistical evidence, then compare the magnitude of lift with the business threshold. Treat segment findings as exploratory unless the experiment was explicitly powered for them.